In [74]:
#Load Model + Scaler
import joblib

def load_model(ticker):
    model = joblib.load(f"../models/{ticker}_model.pkl")
    scaler = joblib.load(f"../models/{ticker}_scaler.pkl")
    return model, scaler

#Fetch Latest stcock Data
import yfinance as yf
import pandas as pd

def fetch_data(ticker):
    df = yf.download(ticker, period="90d", interval = "1d")
    df = df[['Open','High','Low','Close','Volume']]
    return df

In [131]:
#Fetch Latest stcock Data
import yfinance as yf

def fetch_data(ticker):
    df = yf.download(ticker, period="90d", interval="1d", auto_adjust=True)

    # Flatten multiindex columns if needed
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    return df


In [125]:
#Rebuild feature engineering
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

def engineer_features(df):
    # Moving Averages
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()

    # RSI
    df['RSI'] = RSIIndicator(df['Close'], window=14).rsi()

    # MACD
    macd = MACD(df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    df['MACD_hist'] = macd.macd_diff()

    # Bollinger Band width
    indicator_bb = BollingerBands(close=df['Close'], window=20, window_dev=2)
    df['BB_width'] = indicator_bb.bollinger_hband() - indicator_bb.bollinger_lband()

    # Returns
    df['Return'] = df['Close'].pct_change()

    # Momentum
    df['Momentum3'] = df['Close'] / df['Close'].shift(3) - 1
    df['Momentum10'] = df['Close'] / df['Close'].shift(10) - 1

    # Trend
    df['Trend'] = df['Close'] - df['Close'].shift(7)

    # Volatility
    df['Volatility10'] = df['Return'].rolling(10).std()

    # Lags
    df['Lag1'] = df['Close'].shift(1)
    df['Lag5'] = df['Close'].shift(5)

    # Keep ONLY the trained features
    FEATURES = [
        'MA20', 'MA50', 'RSI',
        'MACD', 'MACD_signal', 'MACD_hist',
        'BB_width', 'Return',
        'Momentum3', 'Momentum10', 'Trend', 'Volatility10',
        'Lag1', 'Lag5'
    ]

    df = df[FEATURES]

    # Drop rows with NaN
    #df.dropna(inplace=True)
    df = df.dropna().copy()


    return df


In [145]:
#Build prediction function
import warnings
warnings.filterwarnings("ignore")


import numpy as np
def predict_next_day(ticker):
    model, scaler = load_model(ticker)

    df = fetch_data(ticker)
    df = engineer_features(df)

    if df.shape[0] == 0:
        return ticker, "Not enough data", 0.0

    # Last row as feature vector
    X = df.tail(1).values
  

    # Scale
    X_scaled = scaler.transform(X)

    # Predict
    pred = model.predict(X_scaled)[0]
    prob = model.predict_proba(X_scaled)[0][1]

    direction = "UP" if pred == 1 else "DOWN"

    return direction, prob



In [147]:
#Run predictions
tickers = ["TSLA", "AMZN", "NVDA", "MSFT", "GOOG"]

for ticker in tickers:
    direction, prob = predict_next_day(ticker)
    print(f"{ticker}: {direction} ({prob:.2f} confidence)")


[*********************100%***********************]  1 of 1 completed


TSLA: DOWN (0.17 confidence)


[*********************100%***********************]  1 of 1 completed


AMZN: DOWN (0.42 confidence)


[*********************100%***********************]  1 of 1 completed


NVDA: UP (0.82 confidence)


[*********************100%***********************]  1 of 1 completed


MSFT: UP (0.52 confidence)


[*********************100%***********************]  1 of 1 completed

GOOG: DOWN (0.45 confidence)
